# Exploring Legacy Chains in LangChain

## Install OpenAI, and LangChain dependencies

In [ ]:
!pip install langchain==0.3.11
!pip install langchain-openai==0.2.12
!pip install langchain-community==0.3.11

## Enter Open AI API Key

In [ ]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

## Setup Environment Variables

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY

## Load Connection to LLM

Here we create a connection to ChatGPT to use later in our chains

In [9]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

## Working with LangChain Chains

Using an LLM in isolation is fine for simple applications, but more complex applications require chaining LLMs - either with each other or with other components. Also running on multiple data points can be done easily with chains.

Chain's are the legacy interface for "chained" applications. We define a Chain very generically as a sequence of calls to components, which can include other chains.

Here we use LangChain's Legacy Chains which are going to get ported over to LCEL chains over time

### Working with LLMChain

The most common type of chaining in any LLM application is combining a prompt template with an LLM and optionally an output parser.

An `LLMChain` is a simple chain that adds some functionality around language models. It is used widely throughout LangChain, including in other chains and agents.

An `LLMChain` consists of a PromptTemplate and a language model (either an LLM or chat model). It formats the prompt template using the input key values provided (and also memory key values, if available), passes the formatted string to LLM and returns the LLM output.

__Note:__ `LLMChain` has been deprecated by LangChain in favor of using an LCEL variant, we will be looking at that in a subsequent demo on LCEL chains

In [10]:
reviews = [
    f"""
    Purchased this adorable koala plush toy for my nephew's birthday,
    and he's absolutely smitten with it, carrying it around everywhere he goes.
    The plush is incredibly soft, and the koala's face has an endearing expression.
    However, I did find it a tad on the smaller side given its price point.
    I believe there may be larger alternatives available at a similar price.
    To my delight, it arrived a day earlier than anticipated,
    allowing me to enjoy it briefly before gifting it to him.
    """,
    f"""
    Required a stylish lamp for my office space, and this particular one
    came with added storage at a reasonable price.
    The delivery was surprisingly quick, arriving within just two days.
    However, the pull string for the lamp suffered damage during transit.
    To my relief, the company promptly dispatched a replacement,
    which arrived within a few days. Assembly was a breeze.
    Then, I encountered an issue with a missing component,
    but their support team responded swiftly and provided the missing part.
    It appears to be a commendable company that genuinely values its
    customers and the quality of its products.
    """
    ]

In [11]:
from langchain.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate

prompt = """
            Act as a product review analyst.
            Your task is to generate a short summary of a product
            review from an ecommerce site.

            Generate a summary of the review (max 2 lines)
            Also show both the positives and negatives from the review (max 2 bullets)

            ```{review}```
"""

prompt_template = ChatPromptTemplate.from_template(prompt)
llm_chain = LLMChain(llm=chatgpt, prompt=prompt_template)

/tmp/ipykernel_630326/1585298275.py:16: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=chatgpt, prompt=prompt_template)


In [12]:
reviews[0]

"\n    Purchased this adorable koala plush toy for my nephew's birthday,\n    and he's absolutely smitten with it, carrying it around everywhere he goes.\n    The plush is incredibly soft, and the koala's face has an endearing expression.\n    However, I did find it a tad on the smaller side given its price point.\n    I believe there may be larger alternatives available at a similar price.\n    To my delight, it arrived a day earlier than anticipated,\n    allowing me to enjoy it briefly before gifting it to him.\n    "

In [13]:
result = llm_chain.invoke({'review': reviews[0]})

In [14]:
result

{'review': "\n    Purchased this adorable koala plush toy for my nephew's birthday,\n    and he's absolutely smitten with it, carrying it around everywhere he goes.\n    The plush is incredibly soft, and the koala's face has an endearing expression.\n    However, I did find it a tad on the smaller side given its price point.\n    I believe there may be larger alternatives available at a similar price.\n    To my delight, it arrived a day earlier than anticipated,\n    allowing me to enjoy it briefly before gifting it to him.\n    ",
 'text': '**Summary:** The koala plush toy is soft and adorable, making it a hit with the recipient, though its size may not justify the price.\n\n**Positives:**\n- Incredibly soft and endearing design.\n- Arrived earlier than expected.\n\n**Negatives:**\n- Smaller than anticipated for the price.\n- Potentially larger alternatives available.'}

In [15]:
print(result['text'])

**Summary:** The koala plush toy is soft and adorable, making it a hit with the recipient, though its size may not justify the price.

**Positives:**
- Incredibly soft and endearing design.
- Arrived earlier than expected.

**Negatives:**
- Smaller than anticipated for the price.
- Potentially larger alternatives available.


In [16]:
formatted_reviews = [{'review': review}
                        for review in reviews]

results = llm_chain.map().invoke(formatted_reviews)
len(results)

2

In [17]:
for result in results:
    print(result['text'])
    print()

**Summary:** The koala plush toy is soft and adorable, making it a hit with the recipient, though its size may not justify the price.

**Positives:**
- Incredibly soft and endearing design.
- Arrived earlier than expected.

**Negatives:**
- Smaller than anticipated for the price.
- Potentially larger alternatives available.

**Summary:** A stylish and functional lamp with quick delivery and excellent customer support, though it had initial issues with damaged and missing components.

**Positives:**
- Quick delivery and easy assembly.
- Responsive customer support that promptly addressed issues.

**Negatives:**
- Pull string was damaged during transit.
- Missing component required additional support intervention.



### Combining Multiple Tasks with SequentialChain

The next step after calling a language model is to make a series of calls to a language model. This is particularly useful when you want to take the output from one call and use it as the input to another.

Sequential chains allow you to connect multiple chains and compose them into pipelines that execute some specific scenario.

Here given a few IT Support issues, for each customer issue:
- We want to detect the customer message language
- We want to translate the customer message from the source language to English
- We want the AI to generate a suitable response to the problem in English
- We want to translate this response to the customer language

In [18]:
it_support_queue = [
    "I can't access my email. It keeps showing an error message. Please help.",
    "Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?",
    "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer.",
    "我无法访问公司的网站。每次都显示错误信息。请帮忙解决。"
]

it_support_queue

["I can't access my email. It keeps showing an error message. Please help.",
 'Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?',
 "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer.",
 '我无法访问公司的网站。每次都显示错误信息。请帮忙解决。']

Here we can use `SequentialChain` and chain multiple `LLMChains` where each chain solves one of the tasks using a specific prompt as follows:

In [19]:
# Chain 1: Detect customer message language
prompt1 = """
  Act as a customer support agent.
  For the customer support message delimited below by triple backticks,
  Output the language of the message in one word only, e.g. Spanish

  Customer Message:
  ```{orig_msg}```
"""
prompt_template1 = ChatPromptTemplate.from_template(prompt1)
chain1 = LLMChain(llm=chatgpt, prompt=prompt_template1, output_key="orig_lang")

# Chain 2: Translate Customer Message to English
prompt2 = """
  Act as a customer support agent.
  For the customer message and customer message language delimited below by triple backticks,
  Translate the customer message from the customer message language to English
  if customer message language is not in English,
  else return back the original customer message.

  Customer Message:
  ```{orig_msg}```
  Customer Message Language:
  ```{orig_lang}```
"""
prompt_template2 = ChatPromptTemplate.from_template(prompt2)
chain2 = LLMChain(llm=chatgpt, prompt=prompt_template2, output_key="trans_msg")

# Chain 3: Generate a resolution response in English
prompt3 = """
  Act as a customer support agent.
  For the customer support message delimited below by triple backticks,
  Generate an appropriate resolution response in English.

  Customer Message:
  ```{trans_msg}```
"""
prompt_template3 = ChatPromptTemplate.from_template(prompt3)
chain3 = LLMChain(llm=chatgpt, prompt=prompt_template3, output_key="trans_response")

# Chain 4: Translate resolution response from English to Customer's language
prompt4 = """
  Act as a customer support agent.
  For the customer resolution response and target language delimited below by triple backticks,
  Translate the customer resolution response message from English to the target language
  if target language is not in English,
  else return back the original customer resolution response.

  Customer Resolution Response:
  ```{trans_response}```
  Target Language:
  ```{orig_lang}```
"""
prompt_template4 = ChatPromptTemplate.from_template(prompt4)
chain4 = LLMChain(llm=chatgpt, prompt=prompt_template4, output_key="response")

In [20]:
from langchain.chains import SequentialChain

# combine all chains sequentially
seq_chain = SequentialChain(
    chains=[chain1, chain2, chain3, chain4],
    input_variables=["orig_msg"], # input data
    output_variables=["orig_msg", "orig_lang","trans_msg", "response", "trans_response"], # response data
    verbose=True # set to False to turn of debugging messages
)

In [21]:
support_msgs_formatted = [{'orig_msg': msg} for msg in it_support_queue]
support_msgs_formatted

[{'orig_msg': "I can't access my email. It keeps showing an error message. Please help."},
 {'orig_msg': 'Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?'},
 {'orig_msg': "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer."},
 {'orig_msg': '我无法访问公司的网站。每次都显示错误信息。请帮忙解决。'}]

In [22]:
# run the chain on all the support messages
results = seq_chain.map().invoke(support_msgs_formatted)



> Entering new SequentialChain chain...


> Entering new SequentialChain chain...


> Entering new SequentialChain chain...


> Entering new SequentialChain chain...

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [23]:
results[2]

{'orig_msg': "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer.",
 'orig_lang': 'French',
 'trans_msg': 'The customer message in English is: "My printer is not responding and is no longer printing. I need help to fix it."',
 'response': '```French\nBonjour,\n\nMerci de nous avoir contactés concernant votre problème d\'imprimante. Je suis ici pour vous aider à la remettre en marche.\n\nVoici quelques étapes de dépannage que vous pouvez essayer :\n\n1. **Vérifiez l\'alimentation et les connexions** : Assurez-vous que votre imprimante est branchée et allumée. Vérifiez tous les câbles pour vous assurer qu\'ils sont correctement connectés.\n\n2. **Redémarrez l\'imprimante** : Éteignez votre imprimante, attendez environ 30 secondes, puis rallumez-la. Cela peut souvent résoudre des problèmes temporaires.\n\n3. **Vérifiez les messages d\'erreur** : Regardez sur le panneau d\'affichage de l\'imprimante s\'il y a des messages d\'erreur ou des lumières clignotant

In [24]:
import pandas as pd

df = pd.DataFrame(results)
df

,orig_msg,orig_lang,trans_msg,response,trans_response
0,I can't access my email. It keeps showing an e...,English,I can't access my email. It keeps showing an e...,Subject: Assistance with Email Access\n\nDear ...,Subject: Assistance with Email Access\n\nDear ...
1,Tengo problemas con la VPN. No puedo conectarm...,Spanish,I am having problems with the VPN. I can't con...,```Subject: Asistencia con problemas de conexi...,Subject: Assistance with VPN Connection Issues...
2,Mon imprimante ne répond pas et n'imprime plus...,French,"The customer message in English is: ""My printe...","```French\nBonjour,\n\nMerci de nous avoir con...","Hello,\n\nThank you for reaching out to us reg..."
3,我无法访问公司的网站。每次都显示错误信息。请帮忙解决。,Chinese,I cannot access the company's website. It alwa...,```Chinese\n亲爱的客户，\n\n感谢您与我们联系。很抱歉听到您在访问我们的网站时...,"Dear Customer,\n\nThank you for reaching out t..."


Other popular Legacy Chains include:

- `ConversationChain`
- `ConversationalRetrievalChain`
- `RetrievalQA`

However given LangChain's focus on LCEL Chains, we will be focusing on implementing all of these using LCEL Chains soon.